In [ ]:
# Tratamiento de datos
# ==============================================================================
import numpy as np
import pandas as pd

# Gráficos
# ==============================================================================
import matplotlib.pyplot as plt
plt.style.use('fivethirtyeight')
plt.rcParams['lines.linewidth'] = 1.5
plt.rcParams['font.size'] = 8

# Preprocesado y modelado
# ==============================================================================
import sklearn
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_log_error
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.metrics import mean_squared_error
#from sklearn.metrics import root_mean_squared_log_error
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import train_test_split
from sklearn.model_selection import RepeatedKFold
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import ParameterGrid
from sklearn.model_selection import cross_val_score
from sklearn.inspection import permutation_importance
from sklearn.preprocessing import LabelEncoder
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

from joblib import Parallel, delayed, cpu_count
import optuna

# Configuración warnings
# ==============================================================================
import warnings
optuna.logging.set_verbosity(optuna.logging.WARNING)

print(sklearn.__version__)

In [ ]:
#Cutrez porque el kaggle no me deja usar una versión más moderna
def root_mean_squared_log_error(y_true, y_pred):
    return np.sqrt(mean_squared_log_error(y_true, y_pred))

In [ ]:
# Descarga de datos
# ==============================================================================
train = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/train.csv')
stores = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/stores.csv')
oil = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/oil.csv')
holidays = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/holidays_events.csv')
transactions = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/transactions.csv')

#Mergeo de las STORES en train
train = train.merge(stores, on='store_nbr', how='left')

#Mergeo de OIL en train:
#Original:
#train = train.merge(oil[['date', 'dcoilwtico']], on='date', how='left')
train = train.merge(oil, on='date', how='left')
#Da warning porque el merge crea NaN y eso no es un valor numérico que es lo que vamos a necesitar así que toca formatearlo a valor
#Como lo suyo es cambiar estos valores NaN por predicciones vamos a mergear y refinar todos los datos posibles para a partir de estos hacer la predicción

#Mergeo de HOLIDAYS_EVENTS en train:
#Crea una nueva columna donde todas las filas de feriados tienen el valor 1.
#Esto sirve para que al mergear no solo se pasen las fechas sino que se pasen las fechas y su marca de que son de vacaciones
holidays['holiday_flag'] = 1
#Si filtro las vacaciones para que solo sean nacionales me aseguro de que sea mucho más general.
#Porque no todas las tiendas del mismo país tienen los mismos festivos regionales.
#Y mi predicción es de todo el país no solo de una región.
holidays = holidays[(holidays['type'] == 'Holiday') & (holidays['locale'] == 'National')]
train = train.merge(holidays[['date', 'holiday_flag']], on='date', how='left')

#Mergeo de TRANSACTIONS en train:
train = train.merge(transactions, on=['date', 'store_nbr'], how='left')

#Hasta aquí el mergeo de todos los CSV
#Ahora que ya tenemos todos los datos a nuestra disposición hay que rellenar los datos que nos faltan (transacciones, holiday_flag y dcoilwtico)

#Todo lo que no sean vacaciones que se ponga en '0'
train['holiday_flag'] = train['holiday_flag'].fillna(0)

#Si no hay información de las transacciones suponemos que no hay transacciones (se pone a '0')
train['transactions'] = train['transactions'].fillna(0)

#Se predicen los valores que puede tener dcoilwtico
def rellenarValoresOilConInterativeImputer(df):
    imputer = IterativeImputer(random_state=42)
    #Si sabes cuántas ventas hubo, cuántas transacciones se hicieron, 
    #y cuántos productos estaban en promoción en un día dado, 
    #esa información puede ayudar a inferir cuál fue el precio del petróleo en esa fecha, 
    #cuando está faltante.
    columasParaImputación = ['dcoilwtico', 'transactions', 'sales', 'onpromotion']
    columnasUsadas = [col for col in columasParaImputación if col in df.columns]

    df_temporal  = df[columnasUsadas]
    arrayImputado = imputer.fit_transform(df_temporal)

    df['dcoilwtico'] = arrayImputado[:, columnasUsadas.index('dcoilwtico')]
    return df

train = rellenarValoresOilConInterativeImputer(train)

#Para comprobar que va
print(train.head())
print(train.info())

In [ ]:
#Copia sin valores categóricos para poder hacerlo mediante RF
copiaTrain = train.copy()

# Convertir fechas y extraer variables en las copias
copiaTrain['date'] = pd.to_datetime(copiaTrain['date'])

copiaTrain['day'] = copiaTrain['date'].dt.day
copiaTrain['month'] = copiaTrain['date'].dt.month
copiaTrain['year'] = copiaTrain['date'].dt.year
copiaTrain['weekday'] = copiaTrain['date'].dt.weekday
copiaTrain['is_weekend'] = copiaTrain['weekday'].isin([5, 6]).astype(int)

copiaTrain = copiaTrain.drop(columns=['date'])

# Convertir la columna categórica 'family' a variables dummy
#Lo que hace esto es crear una columna por cada valor y ese valor lo pone a true o false dependiendo de si pertenece o no a esa categoría
#copiaTrain = pd.get_dummies(copiaTrain, columns=['family'], drop_first=True)

#Esto cambia cada valor por uno numérico pero sin eliminar esa forma de categorizar, solo los vuelve numéricos
le = LabelEncoder()
for col in ['family', 'city', 'state', 'type']: #se tienen en cuenta todas las columnas categóricas
    copiaTrain[col] = le.fit_transform(copiaTrain[col])


print("LO IMPORTANTE")
print(copiaTrain.head(3))
print(copiaTrain.info())

In [ ]:
#Se construyen los datos de entrenamiento y los de test.
#La cosa sería mergear todos los csv para así tener más datos a estudiar.
#Pero hacer eso en RF con valores categóricos puede ser una locura.
#Así que voy a probar con solo los de train por ahora.

X = copiaTrain.drop(columns='sales')
y = copiaTrain['sales']
#print(X)
#print(y)
print("Features:")
print(X.columns)
print("Target")
print(y)
#La variable respuesta tiene que ser lo que quiero predecir.
#Según mi problema, la variable respuesta es "unit sales for thousands of items sold at different Favorita stores".
#En mis datos en train tengo una columna llamada "sales" que es la que quiero predecir.
#Por lo tanto, la variable respuesta es "sales".
# El randomForest es un modelo de regresión.
# XGBoost, LightGBM también lo son.
print("Compila")


In [ ]:
# División de los datos en entrenamiento y test
# ==============================================================================
#Drop se encarga de coger las columnas que no quiero.
#En este caso, quiero predecir la columna 'sales', por lo que la variable X_train contendrá todas las columnas excepto 'unit_sales'.
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
print("Compila")
print(X_train, X_val, y_train, y_val)

In [ ]:
# Creación del modelo
# ==============================================================================
modelo = RandomForestRegressor(
    n_estimators=20,     # menos árboles
    max_depth=12,        # reduce la profundidad
    max_samples=0.2,     # usa 20% del dataset por árbol (aún cubre todo en conjunto)
    n_jobs=-1,           # usa los 2 núcleos que da Kaggle
    random_state=42      # Semilla para trackearlo
)
#Me gustaría hacerlo con más arboles y con más profundidad y con más datasets, pero sino no acaba nunca.
print("Compila")
print(X_train.shape)

In [ ]:
# Entrenamiento del modelo
# ==============================================================================
modelo.fit(X_train, y_train)
print("Compila")

In [ ]:
# Error de test del modelo inicial
# (Métrica de validación)
# ==============================================================================
predicciones = modelo.predict(X_val)

#RMSLE
#Sin una máquina virtual no puedo usar esta función porque no tengo la versión suficiente
rmsle = root_mean_squared_log_error(y_val, predicciones)
print(f"El error (RMSLE) en validación es: {rmsle}")
#El error que me da es relativamente alto pero está bastante bien para una primera prueba (1.1085266424812152)
#Error teniendo en cuenta todos los CSV: 0.8577395792374634

#MAE
mae = mean_absolute_error(y_val, predicciones)
print(f"El error (MAE) en validación es: {mae}")
#Error: 100.15685281692434
#Error teniendo en cuenta todos los CSV: 75.07474481159532

#MAPE
mape = mean_absolute_percentage_error(y_val, predicciones)
print(f"El error (MAPE) en validación es: {mape}")
#Error: 3.1189006421862572e+16
#Error teniendo en cuenta todos los CSV: 6833015200759730.0

#MSE
mse = mean_squared_error(y_val, predicciones)
print(f"El error (MSE) en validación es: {mse}")
#Error: 127328.21164493373
#Error teniendo en cuenta todos los CSV: 89014.16747837754

#MSLE
msle = mean_squared_log_error(y_val, predicciones)
print(f"El error (MSLE) en validación es: {msle}")
#Error: 1.2288313170906762
#Error teniendo en cuenta todos los CSV: 0.7357171857904606

